# Order Book Basics

This notebook builds a small visible limit order book, inspects top-of-book metrics, and demonstrates cancellation.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

In [ ]:
from lob_engine.core.matching_engine import MatchingEngine
from lob_engine.core.orders import Order, OrderType, Side
from lob_engine.analytics.microstructure import book_metrics, depth_by_level

engine = MatchingEngine()
orders = [
    Order('B-001', Side.BUY, OrderType.LIMIT, 100, 99.95, timestamp=1),
    Order('B-002', Side.BUY, OrderType.LIMIT, 80, 99.90, timestamp=2),
    Order('A-001', Side.SELL, OrderType.LIMIT, 120, 100.05, timestamp=3),
    Order('A-002', Side.SELL, OrderType.LIMIT, 90, 100.10, timestamp=4),
]
for order in orders:
    engine.process_order(order)

pd.DataFrame([book_metrics(engine.book)])

In [ ]:
ladder = engine.book.top_n_levels(5)
ladder

In [ ]:
fig = px.bar(ladder, x='quantity', y='price', color='side', orientation='h', title='Visible Depth by Price Level')
fig.show()

In [ ]:
from lob_engine.core.orders import CancelRequest

engine.process_cancel(CancelRequest('C-001', timestamp=5, target_order_id='B-002'))
engine.book.top_n_levels(5)